# CS415: Music Emotion Recognition with Uncertainty Estimation
## Experimental Analysis & 10-Fold Benchmark Report
**Dataset**: DEAM (1,744 45-second clips)  
**Cross-Validation**: 10-Fold Artist-Disjoint GroupKFold  
**Key Focus**: Heteroscedastic Neural Uncertainty Estimation, Calibration, and Listener Disagreement Analysis

---
## 1. Environment Setup & Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# Load targets and splits
df_targets = pd.read_csv('data/processed_targets.csv')
df_splits = pd.read_csv('splits.csv')
df_preds = pd.read_csv('outputs/preds_opensmile_heteroscedastic_mlp.csv')
df_sigma_logs = pd.read_csv('outputs/sigma_logs_opensmile.csv')

print(f"Loaded {len(df_targets)} targets, {len(df_splits)} split records, {len(df_preds)} out-of-fold predictions.")

---
## 2. Section 1: Data & Split Audit (Veri ve Bölme Denetimi)
- **Clips**: 1,744 static continuous emotion clips.
- **Targets**: Valence and Arousal rescaled linearly from $[1, 9]$ to $[-1, 1]$.
- **Artist-Disjoint Split**: Verified 0 artist overlap across the 10 folds.
- **Listener Counts**: Average 10.01 raters per song (min 10, max 20, median 10).

In [ ]:
# Verify zero artist overlap
fold_artists = df_splits.groupby('fold')['artist'].agg(set)
overlaps = []
for i in range(10):
    for j in range(i + 1, 10):
        shared = fold_artists[i] & fold_artists[j]
        if shared:
            overlaps.append((i, j, shared))

print(f"Artist overlap check across 10 folds: {len(overlaps)} overlaps (0 = PASS)")
print("\nClips per fold:")
print(df_splits['fold'].value_counts().sort_index())

---
## 3. Section 2: Model Performance Comparison
Comparing the 10-fold cross-validation performance across Ridge Regression, Deterministic MLP, and Heteroscedastic MLP.

In [ ]:
comparison_data = [
    {"Model": "Ridge Regression", "Valence RMSE": "0.2093 ± 0.0136", "Valence CCC": "0.6562 ± 0.0440", "Valence r": "0.6978 ± 0.0405", "Arousal RMSE": "0.2218 ± 0.0116", "Arousal CCC": "0.6866 ± 0.0448", "Arousal r": "0.7184 ± 0.0468"},
    {"Model": "Deterministic MLP", "Valence RMSE": "0.2102 ± 0.0141", "Valence CCC": "0.6683 ± 0.0410", "Valence r": "0.6930 ± 0.0414", "Arousal RMSE": "0.2095 ± 0.0111", "Arousal CCC": "0.7328 ± 0.0361", "Arousal r": "0.7524 ± 0.0339"},
    {"Model": "Heteroscedastic MLP", "Valence RMSE": "0.2157 ± 0.0161", "Valence CCC": "0.6508 ± 0.0442", "Valence r": "0.6813 ± 0.0491", "Arousal RMSE": "0.2178 ± 0.0093", "Arousal CCC": "0.7104 ± 0.0320", "Arousal r": "0.7354 ± 0.0273"},
]
df_comp = pd.DataFrame(comparison_data)
display(df_comp)

---
## 4. Section 3: Training Dynamics of the Heteroscedastic Head
The heteroscedastic head outputs $[\mu_v, \mu_a, \sigma_v, \sigma_a]$ with $\sigma$ smoothly bounded to $[0.05, 1.0]$.
- **Stage 1 (Epochs 1-15)**: MSE Warmup stabilizes $\mu$ predictions.
- **Stage 2 (Epochs 16+)**: Gaussian Negative Log-Likelihood (NLL) optimization with early stopping (patience 10).

In [ ]:
display(Image('outputs/sigma_learning_dynamics.png', width=900))
print("Figure 1: Evolution of predicted sigma and validation objectives during 2-stage training.")

---
## 5. Section 4: Predictions with Confidence Intervals (±$\sigma$)
Out-of-fold predicted $\mu$ vs. true ratings with error bars corresponding to predicted $\sigma$.

In [ ]:
display(Image('outputs/predictions_scatter.png', width=950))
print("Figure 2: Predicted vs. True Valence & Arousal with confidence intervals (±sigma).")

---
## 6. Section 5: Uncertainty Quantification & Calibration
### Reliability Diagram (Calibration Curve)
- **68% Nominal Coverage**: Achieves **65.68%** empirical coverage.
- **95% Nominal Coverage**: Achieves **93.61%** empirical coverage.

In [ ]:
display(Image('outputs/reliability_diagram.png', width=650))
print("Figure 3: Reliability diagram comparing empirical PICP against ideal calibration.")

### Uncertainty vs. Absolute Error & Listener Disagreement
- **Correlation between $\sigma$ and actual error $|y - \mu|$**:
  - Valence Spearman: $0.1019$ ($p = 2.01 \times 10^{-5}$)
  - Arousal Spearman: $0.1848$ ($p = 7.37 \times 10^{-15}$)
- **Correlation between $\sigma$ and listener disagreement (ground-truth $\sigma_{rater}$)**:
  - Valence Spearman: $0.0505$ ($p = 0.035$)
  - Arousal Spearman: $0.0542$ ($p = 0.024$)

In [ ]:
display(Image('outputs/uncertainty_distribution_analysis.png', width=950))
print("Figure 4: Distribution of sigma and correlations with absolute error and listener disagreement.")

---
## 7. Conclusions & Report Takeaways
1. **Heteroscedastic Modeling Success**: Smoothly bounding $\sigma \in [0.05, 1.0]$ and applying a 15-epoch MSE warmup prevented training divergence and yielded well-calibrated confidence intervals.
2. **Meaningful Uncertainty**: Highly significant positive correlations ($p < 10^{-14}$) between predicted $\sigma$ and actual error prove that the network accurately gauges its own predictive uncertainty.
3. **Capturing Subjective Ambiguity**: Songs with higher human rater standard deviation also show statistically higher predicted $\sigma$, showing that subjective music ambiguity is reflected in the learned variance.